# HW 2, Problem 2: Will the zombies win?

We have a problem:

Every day, each person in a particular town is in one of three states: **Healthy**, **Infected** or **Zombie**. A $3\times 3$ matrix $A$ holds the daily (probabilistic) rules on which particular state (H, I, or Z) each person will be in, given the state (H, I or Z) they are in today. We find the percentage of people in each state on dat $n+1$ by **pre-multiplying by $A$ to move the town forward one day**. 

We want to ask **The Big Question:** *after a long time, where does everyone end up?* (Do the zombies eventually die out? Does everyone eventually become a zombie?? Or, do we get an equilibrium, converging to some fraction of the town being zombies???) You will determine the long-term outcome three ways: by waiting it out, by jumping ahead, and with **eigenvalues**.

**The setup.** The vector $x = [H, I, Z]^T$ holds the **fraction** of the total town population that's in each state. Each **column** of $A$ says where one state goes tomorrow, so each column sums to 1, and tomorrow's town is $x_{\text{next}} = A\,x_{\text{today}}$, exactly the matrix-vector product from Unit 2. (Some books store the rules the other way around, with rows as the "from" state, via using matrix $A^T$. Then the update is `x @ P` and you take `eig(P.T)`. The underlying math is the same.)

**ECE Connection:** $x_{\text{n+1}} = A\,x_{\text{n}}$ is a **discrete-time linear system**. Here, matrix $A$ describes **dynamics** on a step-by-step basis (e.g., day-by-day), rather than through continuous time. Understanding and controlling discrete-time dynamics is the core theme in ECE 147B (Digital Control), which is part of the undergraduate controls sequence in our department.

You will also encounter **eigenvalues** if you take ECE 130 (Applied Linear Algebra), and **eigenanalysis** is used in a lot of other ECE domains (e.g., power system analysis, controls and robotics, communication and information theory, circuit and network theory, signal processing and principal component analysis, etc.)

<br>

**How to use this notebook:** run each cell, top to bottom, with the Run button or via **Shift + Enter**. Fill in every `...`, and write your guess at every **Predict first** before running. May take about 30 to 40 minutes. Try to break things in Part 4!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The daily rules. Column = "from" state, row = "to" state.  States: [Healthy, Infected, Zombie]
A = np.array([[0.9, 0.1, 0.1],     # to Healthy:  90% of Healthy stay, 10% of Infected recover, 10% of Zombies are cured
              [0.1, 0.7, 0.0],     # to Infected: 10% of Healthy get infected, 70% of Infected stay
              [0.0, 0.2, 0.9]])    # to Zombie:   20% of Infected turn, 90% of Zombies stay

# The page coordinates of the triangle's corners: column 0 = Healthy, 1 = Infected, 2 = Zombie.
corners = np.array([[0.0, 1.0, 0.5],
                    [0.0, 0.0, np.sqrt(3) / 2]])

def triangle_plot(trajectories, steady=None, title=""):
    '''Each trajectory is a (days+1) x 3 array of [H, I, Z] fractions.  A town is a point in the triangle:
    the closer to a corner, the more of the town is in that state.'''
    plt.figure(figsize=(5.5, 5))
    tri = np.hstack([corners, corners[:, :1]])
    plt.plot(tri[0], tri[1], "k-")
    for name, (cx, cy), dy in zip(["Healthy", "Infected", "Zombie"], corners.T, [-14, -14, 8]):
        plt.annotate(name, (cx, cy), textcoords="offset points", xytext=(0, dy), ha="center")
    for traj in trajectories:
        pts = corners @ np.array(traj).T        # (2 x 3) times (3 x days): every day's page position at once
        plt.plot(pts[0], pts[1], "o-", ms=3, alpha=0.7)
        plt.plot(pts[0, 0], pts[1, 0], "ks")    # black square = where this town started
    if steady is not None:
        s = corners @ steady
        plt.plot(s[0], s[1], "r*", ms=16, label="steady state"); plt.legend(loc="upper right")
    plt.axis("equal"); plt.axis("off"); plt.title(title); plt.show()

def check(ok, hint):
    print("OK - looks right!" if ok else "NOT YET: " + hint)

# Part 1: Read the matrix, then simulate

## Task 1.1: Each column is a promise
Column 2 of `A` (the **Infected** column) contains `[0.1, 0.7, 0.2]`. In words, what happens to an infected person tomorrow? Then check that every column sums to 1 with `A.sum(axis=0)`.

**Why must a column sum to 1?** (You will see what happens when it doesn't, in Part 4.)

**Your answer:**

In [ ]:
column_sums = ...          # the sum of each column of A: A.sum(axis=0)
print(column_sums)
check(np.allclose(column_sums, 1), "each column of A should sum to 1")

## Task 1.2: By hand first
A town of 100 people starts with everyone healthy: `x0 = [100, 0, 0]`.

**Predict first** (no code): how many people are Infected after 1 day? How many are Zombies after 2 days?

**Your prediction:**

Now check with matrix-vector multiplication.

In [ ]:
x0 = np.array([100, 0, 0])
day1 = ...                 # one day later:  A @ x0
day2 = ...                 # two days later: A @ day1
print(day1); print(day2)
check(np.allclose(day1, [90, 10, 0]) and np.allclose(day2, [82, 16, 2]), "day 1 should be [90, 10, 0]")

## Task 1.3: Wait it out (a loop)
Write `simulate(A, x0, days)` that returns the whole history: a list that starts with `x0` and adds one new town per day.

In [ ]:
def simulate(A, x0, days):
    x = np.array(x0, dtype=float)
    history = [x]
    for day in range(days):
        x = ...                # tomorrow's town
        history.append(x)
    return history

history = simulate(A, [1, 0, 0], 50)
for day in [0, 1, 2, 5, 10, 20, 50]:
    print(f"day {day:2d}: Healthy {history[day][0]:.3f}  Infected {history[day][1]:.3f}  Zombie {history[day][2]:.3f}")
check(np.allclose(history[2], [0.82, 0.16, 0.02]), "day 2 should be [0.82, 0.16, 0.02]")

# Part 2: Where does it go?

## Task 2.1: Does the starting distribution (of H, I and Z) matter?
Run three towns: all healthy, all infected, all zombies. The black squares are the starting points.

**Predict first:** will the three paths end up in three different places or the same place?

**Your prediction:**

In [ ]:
starts = [[1, 0, 0], [0, 1, 0], [0, 0, 1]]
paths = [simulate(A, s, 40) for s in starts]
triangle_plot(paths, title="Three towns, 40 days")

**Question 1:** What do you notice? What does it say about whether the **starting** town matters for the **long-term** outcome?

**Your answer:**

## Task 2.2: Jump ahead
Instead of 100 steps in a loop, `np.linalg.matrix_power(A, 100)` multiplies $A$ by itself 100 times: $A^{100}$ takes you 100 days into the future in one line.

In [ ]:
P = ...                    # A multiplied by itself 100 times
print(np.round(P, 3))
check(np.allclose(P @ [1, 0, 0], history[-1], atol=1e-3), "the first column of A^100 should match your day-50 town")

**Question 2:** Look at the three columns of $A^{100}$. What do you notice? What does each column represent (hint: it is $A^{100}$ times a starting town like $[1,0,0]$), and what does it mean that they are all equal?

**Your answer:**

# Part 3: What persists? (Eigenvalue analysis)

A vector $v$ is an **eigenvector** of $A$ if multiplying by $A$ only **stretches** it: $A v = \lambda v$. The stretch factor $\lambda$ is its **eigenvalue**. A town that doesn't change from one day to the next satisfies $A x = x$, which is an eigenvector with $\lambda = 1$. That is the long-term town!

## Task 3.1: The steady state
`np.linalg.eig(A)` returns all the eigenvalues and eigenvectors (the eigenvectors are the **columns** of `vecs`, with arbitrary length and sign). Write `steady_state(A)`: pick the eigenvector whose eigenvalue is largest in absolute value (that is the one with $\lambda = 1$) and **rescale it so its entries sum to 1** (so they are fractions).

In [ ]:
def steady_state(A):
    vals, vecs = np.linalg.eig(A)
    i = np.argmax(np.abs(vals))        # index of the eigenvalue with the largest size
    v = np.real(vecs[:, i])            # its eigenvector (a column of vecs)
    return ...                         # rescale so the entries sum to 1: v / v.sum()

vals, vecs = np.linalg.eig(A)
print("eigenvalues:", np.round(vals, 3))
steady = steady_state(A)
print("steady state [H, I, Z]:", np.round(steady, 3))
check(np.allclose(A @ steady, steady) and np.isclose(steady.sum(), 1), "steady_state(A) should satisfy A @ steady = steady, and sum to 1")
triangle_plot(paths, steady=steady, title="The red star is the steady state")

**Question 3 (by hand):** Show that the town $[3, 1, 2]/6$ is unchanged by one day: compute the first entry of $A\,[3,1,2]/6$ using the first row of $A$. Does it match `steady`? In the long run, what fraction of the town are zombies? Does it match $A^{100}$?

**Your answer:**

## Task 3.2: How long until we get there?
The other two eigenvalues are **smaller than 1**, so those parts of the town **shrink** every day (like $0.8^{\text{day}}$). The biggest of them is the slowest to die, so it sets how long we wait. Count the days until the town is within 0.01 of the steady state, and compare with the estimate $\ln(0.01)/\ln(0.8)$.

In [ ]:
x = np.array([1.0, 0, 0])
days = 0
while np.linalg.norm(x - steady) >= 0.01:
    x = ...                  # one more day
    days += 1
print(days, "days to get within 0.01 of the steady state")
print("estimate from the second eigenvalue:", round(np.log(0.01) / np.log(0.8), 1), "days")
check(20 <= days <= 25, "it should take about 22 days")

**Question 4:** Which eigenvalue controls how long the wait is, 0.8 or 0.7? What would happen to the waiting time if that eigenvalue were **0.95**?

**Your answer:**

# Part 4: Change the world, then break it

The function `make_A(infect, cure)` rebuilds the matrix with your choice of how fast healthy people get infected and how fast zombies are cured (it keeps every column summing to 1).

In [ ]:
def make_A(infect, cure):
    return np.array([[1 - infect, 0.1, cure],
                     [infect,     0.7, 0.0],
                     [0.0,        0.2, 1 - cure]])

print("long-run zombie fraction:")
print("infect \\ cure   0.1    0.2    0.4")
for infect in [0.1, 0.05, 0.02]:
    row = [steady_state(make_A(infect, cure))[2] for cure in [0.1, 0.2, 0.4]]
    print(f"   {infect:<10}  " + "  ".join(f"{z:.3f}" for z in row))

**Task 4.1: Your policy.** Which is more effective for reducing the zombie share: **halving** the infection rate (0.1 to 0.05) or **doubling** the cure rate (0.1 to 0.2)? Pick your own `infect` and `cure` below to bring the long-run zombie fraction **under 10%**, and report them.

In [ ]:
my_infect = ...            # your choice
my_cure = ...              # your choice
print("long-run zombie fraction:", round(steady_state(make_A(my_infect, my_cure))[2], 3))

**Your answer** (which lever works better, and your chosen numbers):

**Break it #1: no cure.** **Predict first:** with `cure = 0`, who wins in the long run?

In [ ]:
A0 = make_A(0.1, 0.0)
print("steady state:", np.round(steady_state(A0), 3))
print("eigenvalue sizes:", np.round(np.sort(np.abs(np.linalg.eigvals(A0))), 3))

**Break it #1 question:** Why do the zombies take over completely? (Look at the Zombie column of `A0`: where can a zombie go?) And what happened to the second-largest eigenvalue compared with before, so is the takeover faster or slower than the 22 days we found?

**Your answer:**

**Break it #2: a leak.** We change one entry so the first column sums to 0.95: 5% of healthy people simply vanish every day.

In [ ]:
B = A.copy()
B[0, 0] = 0.85
x = np.array([1.0, 0, 0])
for day in range(50):
    x = B @ x
print("town after 50 days:", np.round(x, 3), " total =", round(x.sum(), 3))
print("largest eigenvalue size:", round(np.max(np.abs(np.linalg.eigvals(B))), 3))

**Break it #2 question:** What happened to the total, and what does the largest eigenvalue being a little less than 1 tell you? So what does an eigenvalue of exactly 1 mean for a matrix like $A$?

**Your answer:**

# What you found
- The steady state is Healthy **_____**, Infected **_____**, Zombie **_____**.
- The eigenvalues of $A$ are **_____**, and the town gets within 1% of the steady state in about **_____** days.
- My policy: `infect` = **_____**, `cure` = **_____**, giving **_____** zombies in the long run.
- One sentence: in your own words, what is an eigenvector with eigenvalue 1 for a system like this?

**Before you submit:** use **Kernel → Restart & Run All** and make sure nothing gives an error. Save and submit on Gradescope.